# PARIS API – Gemeinderat der Stadt Zürich
## Tutorial: Parlamentsdaten mit Python und pandas abfragen

Dieses Tutorial zeigt, wie man die **OGD-Schnittstelle (PARIS API)** des Gemeinderats der Stadt Zürich mit Python und **pandas** nutzt.

### API-Überblick

| Eigenschaft | Wert |
|---|---|
| Basis-URL | `https://www.gemeinderat-zuerich.ch/api/` |
| Protokoll | HTTPS / REST (nur lesend) |
| Datenformat | **XML** |
| Authentifizierung | Keine – frei nutzbar |
| Sprache | Nur `de-CH` |
| Abfragesprache | CQL (Contextual Query Language) |

### URL-Schema

```
https://www.gemeinderat-zuerich.ch/api/{index}/{funktion}?q={cql-query}&l=de-CH&s={start}&m={max}
```

| Teil | Bedeutung |
|---|---|
| `{index}` | Datenentitätstyp, z.B. `geschaeft`, `kontakt`, `sitzung` |
| `{funktion}` | `searchdetails` oder `schema` |
| `q` | CQL-Abfrage (zwingend bei `searchdetails`) |
| `l` | Sprache – immer `de-CH` |
| `s` | Start-Treffer (Standard: 1) |
| `m` | Max. Treffer (Standard: 1000, Max: 1000) |

### Verfügbare Indizes

| Index | Inhalt |
|---|---|
| `geschaeft` | Ratsgeschäfte (Motionen, Postulate, Anfragen, …) |
| `kontakt` | Ratsmitglieder (aktiv & ehemalig) |
| `sitzung` | Ratssitzungen inkl. Traktanden & Wortmeldungen |
| `behoerdenmandat` | Mandate (wer ist/war in welchem Gremium) |
| `abstimmung` | Abstimmungsdaten (ab Mai 2023) |
| `wortmeldung` | Wortmeldungen (ab Mai 2023) |
| `gremiumdetail` | Gremien mit aktuellen Mitgliedern |
| `gremiumsuebersicht` | Alle Gremien (Kommissionen, Fraktionen, …) |
| `dokument` | Dokument-Metadaten |
| `ratspost` | Ratspost-Einträge |
| `partei` | Partei-Stammdaten |
| `departement` | Departement-Stammdaten |

---

## Inhaltsverzeichnis

1. [Setup & Hilfsfunktionen](#1-setup--hilfsfunktionen)
2. [Schema eines Index abrufen](#2-schema-eines-index-abrufen)
3. [CQL-Abfragesprache](#3-cql-abfragesprache)
4. [Ratsmitglieder abfragen (Index: KONTAKT)](#4-ratsmitglieder-abfragen-index-kontakt)
   - [4.1 Alle aktiven Ratsmitglieder](#41-alle-aktiven-ratsmitglieder)
   - [4.2 Analyse: Sitzverteilung nach Partei](#42-analyse-sitzverteilung-nach-partei)
   - [4.3 Einzelne Person suchen](#43-einzelne-person-suchen)
   - [4.4 Mitglieder einer Fraktion](#44-mitglieder-einer-fraktion)
   - [4.5 Altersstruktur des Gemeinderats](#45-altersstruktur-des-gemeinderats)
5. [Ratsgeschäfte abfragen (Index: GESCHAEFT)](#5-ratsgeschäfte-abfragen-index-geschaeft)
   - [5.1 Einzelnes Geschäft nach GR-Nummer](#51-einzelnes-geschäft-nach-gr-nummer)
   - [5.2 Alle Geschäfte eines Jahrgangs](#52-alle-geschäfte-eines-jahrgangs)
   - [5.3 Geschäfte nach Stichwort im Titel suchen](#53-geschäfte-nach-stichwort-im-titel-suchen)
   - [5.4 Kombinierte CQL-Abfrage: Dringliche Motionen 2022–2024](#54-kombinierte-cql-abfrage-dringliche-motionen-20222024)
   - [5.5 Geschäfte nach Departement filtern](#55-geschäfte-nach-departement-filtern)
   - [5.6 Analyse: Geschäfte pro Jahr (mehrere Jahrgänge)](#56-analyse-geschäfte-pro-jahr-mehrere-jahrgänge)
6. [Ratssitzungen abfragen (Index: SITZUNG)](#6-ratssitzungen-abfragen-index-sitzung)
   - [6.1 Sitzungen nach Datum](#61-sitzungen-nach-datum)
   - [6.2 Kommende Sitzungen](#62-kommende-sitzungen)
7. [Behördenmandate abfragen (Index: BEHOERDENMANDAT)](#7-behördenmandate-abfragen-index-behoerdenmandat)
8. [Dokumente abfragen & herunterladen](#8-dokumente-abfragen--herunterladen)
9. [Gremien abfragen (Index: GREMIUMSUEBERSICHT)](#9-gremien-abfragen-index-gremiumsuebersicht)
10. [Paginierung: Grosse Datenmenge laden](#10-paginierung-grosse-datenmenge-laden)
11. [Praxisbeispiel: Wer hat wie viele Vorstösse eingereicht?](#11-praxisbeispiel-wer-hat-wie-viele-vorstösse-eingereicht)
12. [Zusammenfassung](#12-zusammenfassung)


## 1. Setup & Hilfsfunktionen

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from urllib.parse import quote
from datetime import date

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)

BASE_URL = "https://www.gemeinderat-zuerich.ch/api"
LANG = "de-CH"

# XML-Namespaces
NS_RESPONSE = "http://www.cmiag.ch/cdws/searchDetailResponse"


def paris_search(
    index: str,
    query: str,
    start: int = 1,
    max_rows: int = 100,
) -> ET.Element:
    """Sendet eine searchdetails-Anfrage und gibt das Root-XML-Element zurück."""
    url = f"{BASE_URL}/{index}/searchdetails"
    params = {"q": query, "l": LANG, "s": start, "m": max_rows}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    # Prüfen ob die Antwort valides XML enthält (API gibt bei Fehlern HTML zurück)
    if not response.content.lstrip().startswith(b"<"):
        raise ValueError(
            f"API-Fehler – keine XML-Antwort erhalten.\n"
            f"Antwort (erste 300 Zeichen):\n{response.text[:300]}"
        )
    try:
        return ET.fromstring(response.content)
    except ET.ParseError as exc:
        raise ValueError(
            f"XML-Parsefehler: {exc}\n"
            f"Hinweis: Die API gibt bei ungültigen Abfragen eine HTML-Fehlerseite zurück.\n"
            f"Antwort (erste 300 Zeichen):\n{response.text[:300]}"
        ) from exc


def paris_schema(index: str) -> ET.Element:
    """Ruft das XML-Schema eines Index ab."""
    url = f"{BASE_URL}/{index}/schema"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return ET.fromstring(response.content)


def get_num_hits(root: ET.Element) -> int:
    """Liest die Gesamttrefferanzahl aus dem Response-Header."""
    return int(root.attrib.get("numHits", 0))


def get_hits(root: ET.Element) -> list:
    """Gibt alle Hit-Elemente als Liste zurück."""
    return root.findall(f"{{{NS_RESPONSE}}}Hit")


def xml_to_dict(element: ET.Element, prefix: str = "") -> dict:
    """Konvertiert ein XML-Element rekursiv in ein flaches Dictionary."""
    result = {}
    tag = element.tag.split("}", 1)[-1]  # Namespace entfernen
    key = f"{prefix}{tag}" if prefix else tag

    # Attribute des Elements
    for attr_name, attr_val in element.attrib.items():
        result[f"{key}@{attr_name}"] = attr_val

    children = list(element)
    if children:
        for child in children:
            child_dict = xml_to_dict(child, prefix=f"{key}/")
            for k, v in child_dict.items():
                # Bei mehrfachen Kinderelementen gleichen Namens: kommasepariert anhängen
                if k in result:
                    result[k] = f"{result[k]}, {v}"
                else:
                    result[k] = v
    elif element.text and element.text.strip():
        result[key] = element.text.strip()

    return result


def hits_to_dataframe(hits: list, data_tag: str = None) -> pd.DataFrame:
    """
    Wandelt eine Liste von Hit-Elementen in ein pandas DataFrame um.
    data_tag: optionaler Tag-Name des Daten-Elements im Hit (z.B. 'Geschaeft').
              Wenn None, wird das erste Nicht-Snippet-Element verwendet.
    """
    rows = []
    for hit in hits:
        row = {
            "_Guid": hit.attrib.get("Guid"),
            "_SEQ": hit.attrib.get("SEQ"),
        }
        # Daten-Element finden (erstes Kind, das kein Snippet ist)
        for child in hit:
            child_tag = child.tag.split("}", 1)[-1]
            if child_tag == "Snippet":
                continue
            if data_tag is None or child_tag == data_tag:
                data_dict = xml_to_dict(child)
                row.update(data_dict)
                break
        rows.append(row)
    return pd.DataFrame(rows)


def load_all_pages(
    index: str,
    query: str,
    page_size: int = 500,
    max_total: int = 5000,
) -> pd.DataFrame:
    """
    Lädt alle Treffer einer Abfrage paginiert und gibt ein DataFrame zurück.
    Stoppt bei max_total Treffern.
    """
    frames = []
    start = 1

    # Erste Seite für Gesamtanzahl
    root = paris_search(index, query, start=start, max_rows=page_size)
    total = get_num_hits(root)
    total_to_load = min(total, max_total)
    print(f"Gesamt: {total:,} Treffer, lade {total_to_load:,}")

    hits = get_hits(root)
    frames.append(hits_to_dataframe(hits))
    start += page_size

    while start <= total_to_load:
        root = paris_search(index, query, start=start, max_rows=page_size)
        hits = get_hits(root)
        frames.append(hits_to_dataframe(hits))
        start += page_size
        print(f"  Geladen: {min(start - 1, total_to_load):,} / {total_to_load:,}", end="\r")

    print()
    return pd.concat(frames, ignore_index=True)


print("Setup abgeschlossen. Basis-URL:", BASE_URL)
print(f"Aktuelles Jahr: {date.today().year}")

---
## 2. Schema eines Index abrufen

Mit der Funktion `schema` können die verfügbaren Felder und Suchparameter eines Index ermittelt werden.
Das hilft zu verstehen, nach welchen Feldern gesucht werden kann.

In [ ]:
def get_search_fields(index: str) -> pd.DataFrame:
    """Gibt die Suchfelder eines Index als DataFrame zurück."""
    schema = paris_schema(index)
    # SearchField-Elemente aus der Annotation extrahieren
    rows = []
    for elem in schema.iter():
        tag = elem.tag.split("}", 1)[-1]
        if tag == "SearchField":
            xsi_type = elem.attrib.get(
                "{http://www.w3.org/2001/XMLSchema-instance}type", ""
            )
            rows.append({
                "Feldname": elem.attrib.get("Name"),
                "Typ": xsi_type.replace("SearchField", ""),
                "BoostFactor": elem.attrib.get("BoostFactor"),
            })
    return pd.DataFrame(rows)


# Suchfelder des Index GESCHAEFT
df_schema_geschaeft = get_search_fields("geschaeft")
print(f"Suchfelder des Index GESCHAEFT ({len(df_schema_geschaeft)} Felder):")
df_schema_geschaeft

In [ ]:
# Suchfelder des Index KONTAKT
df_schema_kontakt = get_search_fields("kontakt")
print(f"Suchfelder des Index KONTAKT ({len(df_schema_kontakt)} Felder):")
df_schema_kontakt

---
## 3. CQL-Abfragesprache

Die API nutzt **CQL (Contextual Query Language)** für Suchabfragen.

### Aufbau eines Suchkriteriums

```
Feldname  Relation  "Suchbegriff"
  GRNr      any      "2023/100"
```

### Relationen

| Relation | Typ | Bedeutung |
|---|---|---|
| `any` | Text | Mindestens ein Wort enthalten |
| `all` | Text | Alle Wörter enthalten |
| `adj` | Text | Exakte Wortfolge (Phrase) |
| `=` `>` `>=` `<` `<=` | Datum/Zahl/Bool | Vergleich |

### Boolsche Operatoren

```cql
name any "Meier" AND vorname any "Anna"   # UND
name any "Meier" OR  name any "Müller"   # ODER
name any "Meier" NOT vorname any "Max"   # UND NICHT
```

### Sortierung

```cql
vorname any "Rolf" sortBy partei/sort.ascending
```

### Alle Objekte eines Index

```cql
seq>0
```


---
## 4. Ratsmitglieder abfragen (Index: KONTAKT)

### 4.1 Alle aktiven Ratsmitglieder

Aktive Ratsmitglieder werden in zwei Schritten ermittelt:

1. **Kein Enddatum:** Das eingebettete Behördenmandat hat `Dauer/End` mit `xsi:nil="true"` –
   entspricht der API-Logik `Dauer_end > "9999-12-31 00:00:00"` im Index `behoerdenmandat`.
2. **Funktion filtern:** Nicht alle Mandate sind ordentliche Sitze – Ersatzmitglieder
   und andere Funktionen müssen ausgeschlossen werden.

In [ ]:
# Schritt 1: Aktive Gemeinderat-Mandate direkt aus behoerdenmandat-Index laden
# Dauer_end > "9999-12-31 00:00:00" entspricht offenem Enddatum (xsi:nil="true")
root_kontakte = paris_search(
    "behoerdenmandat",
    'Gremium any "Gemeinderat" AND Dauer_end > "9999-12-31 00:00:00"',
    max_rows=200
)
df_kontakte = hits_to_dataframe(get_hits(root_kontakte))
print(f"Aktive Ratsmitglieder (alle Funktionen): {len(df_kontakte)}")
df_kontakte.head(3)

In [ ]:
# Schritt 2: Funktion-Verteilung anzeigen
funktion_col = next((c for c in df_kontakte.columns if "Funktion" in c), None)

if funktion_col:
    print(f"Funktion-Spalte: {funktion_col}")
    print("\nVerteilung:")
    print(df_kontakte[funktion_col].value_counts().to_string())

    # Visualisierung
    funktion_counts = (
        df_kontakte[funktion_col]
        .value_counts()
        .reset_index()
        .rename(columns={funktion_col: "Funktion", "count": "Anzahl"})
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(funktion_counts["Funktion"], funktion_counts["Anzahl"], color="steelblue")
    ax.set_xlabel("Anzahl")
    ax.set_title("Aktive Gemeinderat-Mandate nach Funktion")
    ax.bar_label(ax.containers[0], padding=4)
    plt.tight_layout()
    plt.show()

    # Schritt 3: Nur ordentliche Sitze – Ersatzmitglieder ausschliessen
    df_kontakte_aktiv = df_kontakte[
        ~df_kontakte[funktion_col].str.contains("Ersatz", na=False)
    ].copy()
    print(f"\nAktive Ratsmitglieder (ohne Ersatz): {len(df_kontakte_aktiv)}")
    df_kontakte_aktiv.head(5)
else:
    print("Funktion-Spalte nicht gefunden. Verfügbare Spalten:", df_kontakte.columns.tolist())
    df_kontakte_aktiv = df_kontakte.copy()

In [ ]:
# Spalten erkunden: welche persönlichen Felder liefert behoerdenmandat?
print(f"Spalten in df_kontakte_aktiv ({len(df_kontakte_aktiv.columns)} total):")
for c in sorted(df_kontakte_aktiv.columns):
    print(" ", c)

In [ ]:
# Relevante Felder automatisch aus behoerdenmandat-Spalten erkennen
def find_col(df, *keywords):
    """Gibt die erste Spalte zurück, die alle Schlüsselwörter enthält."""
    for c in df.columns:
        if all(kw.lower() in c.lower() for kw in keywords):
            return c
    return None

name_col      = find_col(df_kontakte_aktiv, "Name")
vorname_col   = find_col(df_kontakte_aktiv, "Vorname")
partei_col    = find_col(df_kontakte_aktiv, "Partei")
jahrgang_col  = find_col(df_kontakte_aktiv, "Jahrgang")
wahlkreis_col = find_col(df_kontakte_aktiv, "Wahlkreis")
wohnkreis_col = find_col(df_kontakte_aktiv, "Wohnkreis")
geschlecht_col = find_col(df_kontakte_aktiv, "Geschlecht")

print(f"Name:       {name_col}")
print(f"Vorname:    {vorname_col}")
print(f"Partei:     {partei_col}")
print(f"Jahrgang:   {jahrgang_col}")
print(f"Wahlkreis:  {wahlkreis_col}")
print(f"Wohnkreis:  {wohnkreis_col}")
print(f"Geschlecht: {geschlecht_col}")

basis_cols = [c for c in [name_col, vorname_col, partei_col, geschlecht_col,
                            jahrgang_col, wahlkreis_col, wohnkreis_col]
              if c is not None]

df_mitglieder = df_kontakte_aktiv[basis_cols].copy()
df_mitglieder.columns = [c.rsplit("/", 1)[-1] for c in basis_cols]
print(f"\nSpalten in df_mitglieder: {df_mitglieder.columns.tolist()}")
df_mitglieder.head(10)

### 4.2 Analyse: Sitzverteilung nach Partei


In [ ]:
# Parteispalte im umbenannten DataFrame ermitteln
partei_kurz = partei_col.split("/")[-1] if partei_col else None

if partei_kurz and partei_kurz in df_mitglieder.columns:
    parteien = (
        df_mitglieder[partei_kurz]
        .value_counts()
        .reset_index()
        .rename(columns={partei_kurz: "Partei", "count": "Sitze"})
    )
    print(parteien.to_string(index=False))
else:
    print("Keine Parteispalte gefunden. Verfügbare Spalten:", df_mitglieder.columns.tolist())
    parteien = None


In [ ]:
# Kreisdiagramm Parteienverteilung
if parteien is not None:
    fig, ax = plt.subplots(figsize=(8, 8))
    colors = plt.cm.tab20.colors
    wedges, texts, autotexts = ax.pie(
        parteien["Sitze"],
        labels=parteien["Partei"],
        autopct=lambda p: f"{p:.1f}%\n({int(round(p * parteien['Sitze'].sum() / 100))})" if p > 3 else "",
        colors=colors,
        startangle=90,
    )
    ax.set_title("Sitzverteilung Gemeinderat Zürich (aktive Mitglieder)")
    plt.tight_layout()
    plt.show()
else:
    print("Keine Parteidaten für Grafik verfügbar.")


### 4.3 Einzelne Person suchen


In [ ]:
# Kontakt nach Name und Vorname suchen
root_person = paris_search(
    "kontakt",
    'Name any "Müller" AND Vorname any "Peter"',
    max_rows=10
)
print(f"Treffer: {get_num_hits(root_person)}")

df_person = hits_to_dataframe(get_hits(root_person))
df_person.head()

### 4.4 Mitglieder einer Fraktion


In [ ]:
# Alle SP-Ratsmitglieder
root_sp = paris_search("kontakt", 'Fraktion any "SP"', max_rows=100)
print(f"SP-Fraktion: {get_num_hits(root_sp)} Mitglieder")

df_sp = hits_to_dataframe(get_hits(root_sp))
name_cols = [c for c in df_sp.columns if "Name" in c or "Vorname" in c][:4]
df_sp[name_cols].head(10)

### 4.5 Altersstruktur des Gemeinderats


In [ ]:
if jahrgang_col and jahrgang_col in df_kontakte_aktiv.columns:
    jg_selected = [jahrgang_col]
    if partei_col and partei_col in df_kontakte_aktiv.columns:
        jg_selected.append(partei_col)
    if geschlecht_col and geschlecht_col in df_kontakte_aktiv.columns:
        jg_selected.append(geschlecht_col)

    df_mitglieder_jg = df_kontakte_aktiv[jg_selected].copy()
    df_mitglieder_jg[jahrgang_col] = pd.to_numeric(df_mitglieder_jg[jahrgang_col], errors="coerce")
    df_mitglieder_jg = df_mitglieder_jg.dropna(subset=[jahrgang_col])
    df_mitglieder_jg["Alter"] = date.today().year - df_mitglieder_jg[jahrgang_col]

    print(f"Durchschnittsalter: {df_mitglieder_jg['Alter'].mean():.1f} Jahre")
    print(f"Jüngstes Mitglied:  {int(df_mitglieder_jg['Alter'].min())} Jahre")
    print(f"Ältestes Mitglied:  {int(df_mitglieder_jg['Alter'].max())} Jahre")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_mitglieder_jg["Alter"], bins=15, color="steelblue", edgecolor="white")
    ax.axvline(df_mitglieder_jg["Alter"].mean(), color="coral", linewidth=2,
               label=f"Durchschnitt: {df_mitglieder_jg['Alter'].mean():.1f} J.")
    ax.set_xlabel("Alter")
    ax.set_ylabel("Anzahl Ratsmitglieder")
    ax.set_title("Altersverteilung im Gemeinderat Zürich")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Jahrgang-Spalte nicht verfügbar im behoerdenmandat-Index.")
    print("Tipp: Für Jahrgang kontakt-Index direkt abfragen oder behoerdenmandat mit kontakt verknüpfen.")

---
## 5. Ratsgeschäfte abfragen (Index: GESCHAEFT)

### 5.1 Einzelnes Geschäft nach GR-Nummer


In [ ]:
# Ein spezifisches Geschäft nach GR-Nummer suchen
root = paris_search("geschaeft", 'GRNr any "2023/100"')

print(f"Treffer: {get_num_hits(root)}")
hits = get_hits(root)

# Rohe XML-Struktur eines Treffers anzeigen
if hits:
    print("\nXML-Struktur des ersten Treffers (Auszug):")
    print(ET.tostring(hits[0], encoding="unicode")[:2000])

In [ ]:
# Als DataFrame ausgeben
df_geschaeft = hits_to_dataframe(hits)
# Relevante Spalten anzeigen
cols = [c for c in df_geschaeft.columns if any(
    kw in c for kw in ["GRNr", "Titel", "Geschaeftsart", "PendentBei", "Beginn", "Dringlich"]
)]
df_geschaeft[cols].T

### 5.2 Alle Geschäfte eines Jahrgangs


In [ ]:
# Alle Geschäfte im Jahr 2023 (Wildcard mit *)
root_2023 = paris_search("geschaeft", 'GRNr any "2023/*"', max_rows=1000)
num_hits = get_num_hits(root_2023)
print(f"Geschäfte 2023: {num_hits}")

df_2023 = hits_to_dataframe(get_hits(root_2023))
print(f"Spalten: {df_2023.columns.tolist()[:10]}...")
df_2023.head(3)

In [ ]:
# Geschäftsarten analysieren
art_col = "Geschaeft/Geschaeftsart"
if art_col in df_2023.columns:
    df_arten = (
        df_2023[art_col]
        .value_counts()
        .reset_index()
        .rename(columns={art_col: "Geschäftsart", "count": "Anzahl"})
    )
    print(df_arten.to_string(index=False))
else:
    print("Verfügbare Spalten:", [c for c in df_2023.columns if "Geschaeft" in c][:15])

In [ ]:
# Visualisierung: Geschäftsarten 2023
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df_arten["Geschäftsart"], df_arten["Anzahl"], color="steelblue")
ax.set_xlabel("Geschäftsart")
ax.set_ylabel("Anzahl")
ax.set_title("Gemeinderatsgeschäfte 2023 nach Geschäftsart")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 5.3 Geschäfte nach Stichwort im Titel suchen


In [ ]:
# Geschäfte zum Thema «Klimaschutz»
root_klima = paris_search("geschaeft", 'Titel any "Klimaschutz"', max_rows=50)
print(f"Treffer: {get_num_hits(root_klima)}")

df_klima = hits_to_dataframe(get_hits(root_klima))

# Relevante Spalten
cols = ["Geschaeft/GRNr", "Geschaeft/Titel", "Geschaeft/Geschaeftsart",
        "Geschaeft/PendentBei", "Geschaeft/Beginn"]
cols_available = [c for c in cols if c in df_klima.columns]
df_klima[cols_available].head(10)

### 5.4 Kombinierte CQL-Abfrage: Dringliche Motionen 2022–2024


In [ ]:
# Dringliche Motionen in mehreren Jahren
query = (
    '(GRNr any "2022/*" OR GRNr any "2023/*" OR GRNr any "2024/*") '
    'AND Geschaeftsart any "Motion" '
    'AND Dringlich = true'
)

root_dringl = paris_search("geschaeft", query, max_rows=200)
print(f"Dringliche Motionen 2022-2024: {get_num_hits(root_dringl)}")

df_dringl = hits_to_dataframe(get_hits(root_dringl))
cols_available = [c for c in ["Geschaeft/GRNr", "Geschaeft/Titel", "Geschaeft/Beginn"] 
                  if c in df_dringl.columns]
df_dringl[cols_available]

### 5.5 Geschäfte nach Departement filtern


In [ ]:
# Alle Geschäfte des Finanzdepartements im Jahr 2024
query = 'GRNr any "2024/*" AND Departement any "Finanzdepartement"'
root_fd = paris_search("geschaeft", query, max_rows=200)
print(f"Finanzdepartement 2024: {get_num_hits(root_fd)} Geschäfte")

df_fd = hits_to_dataframe(get_hits(root_fd))
cols_available = [c for c in ["Geschaeft/GRNr", "Geschaeft/Titel", "Geschaeft/Geschaeftsart"]
                  if c in df_fd.columns]
df_fd[cols_available].head(10)

### 5.6 Analyse: Geschäfte pro Jahr (mehrere Jahrgänge)


In [ ]:
# Anzahl Geschäfte pro Jahr (2018 bis aktuelles Jahr)
aktuelles_jahr = date.today().year
jahre = range(2018, aktuelles_jahr + 1)
rows_jahre = []

for jahr in jahre:
    root = paris_search("geschaeft", f'GRNr any "{jahr}/*"', max_rows=1)
    rows_jahre.append({"Jahr": jahr, "Anzahl Geschäfte": get_num_hits(root)})

df_jahre = pd.DataFrame(rows_jahre)
df_jahre

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df_jahre["Jahr"].astype(str), df_jahre["Anzahl Geschäfte"], color="steelblue")
ax.set_xlabel("Jahr")
ax.set_ylabel("Anzahl Geschäfte")
ax.set_title(f"Gemeinderatsgeschäfte pro Jahrgang (2018–{aktuelles_jahr})")
ax.bar_label(ax.containers[0], padding=3)
plt.tight_layout()
plt.show()

---
## 6. Ratssitzungen abfragen (Index: SITZUNG)

### 6.1 Sitzungen nach Datum

In [ ]:
# Sitzungen im Jahr 2024 (Datumsbereich)
query_sitzungen = (
    'Sitzungsdatum_start >= "2024-01-01 00:00:00" '
    'AND Sitzungsdatum_start <= "2024-12-31 23:59:59" '
    'sortBy Sitzungsdatum_start/sort.ascending'
)

root_sitzungen = paris_search("sitzung", query_sitzungen, max_rows=100)
print(f"Sitzungen 2024: {get_num_hits(root_sitzungen)}")

df_sitzungen = hits_to_dataframe(get_hits(root_sitzungen))
df_sitzungen.head(3)

In [ ]:
# Relevante Sitzungsfelder extrahieren
sitzung_cols = [c for c in df_sitzungen.columns
                if any(kw in c for kw in ["Titel", "Datum", "Nummer"])
                and c.count("/") <= 2]
print("Sitzungs-Spalten:", sitzung_cols)

if sitzung_cols:
    df_sitzungen[sitzung_cols].head(10)

### 6.2 Kommende Sitzungen

In [ ]:
heute = date.today().strftime("%Y-%m-%d 00:00:00")

root_naechste = paris_search(
    "sitzung",
    f'Sitzungsdatum_start > "{heute}" sortBy Sitzungsdatum_start/sort.ascending',
    max_rows=10
)
print(f"Nächste Sitzungen ab heute: {get_num_hits(root_naechste)}")

df_naechste = hits_to_dataframe(get_hits(root_naechste))
sitzung_cols = [c for c in df_naechste.columns if c.count("/") <= 2]
df_naechste[sitzung_cols[:8]].head()

---
## 7. Behördenmandate abfragen (Index: BEHOERDENMANDAT)

Über Behördenmandate kann ermittelt werden, wer aktuell oder in einer Zeitspanne ein Mandat in einem bestimmten Gremium innehat.

In [ ]:
# Alle Behördenmandate im Gemeinderat laden
# Hinweis: Das Feld 'Dauer' existiert nicht im Index; alle Mandate werden geladen
# und können anschliessend Python-seitig nach offenem Ende gefiltert werden.
query_mandate = 'Gremium any "Gemeinderat"'

root_mandate = paris_search("behoerdenmandat", query_mandate, max_rows=200)
print(f"Mandate Gemeinderat: {get_num_hits(root_mandate)}")

df_mandate = hits_to_dataframe(get_hits(root_mandate))
df_mandate.head(3)


In [ ]:
# Funktion (Präsidium, Mitglied, etc.) analysieren
funktion_col = next((c for c in df_mandate.columns if "Funktion" in c), None)
if funktion_col:
    print(df_mandate[funktion_col].value_counts().to_string())
else:
    print("Verfügbare Spalten:", [c for c in df_mandate.columns if c.count("/") <= 2][:10])

---
## 8. Dokumente abfragen & herunterladen

Dokumente (PDFs) können über die API in zwei Schritten abgerufen werden:
1. Metadaten per `dokument/searchdetails`
2. Binärdatei per `https://www.gemeinderat-zuerich.ch/dokumente/{file-id}`

In [ ]:
# Zuerst: Dokument-GUID aus einem Geschäft holen
root_gs = paris_search("geschaeft", 'GRNr any "2023/100"', max_rows=1)
hits_gs = get_hits(root_gs)

if hits_gs:
    xml_str = ET.tostring(hits_gs[0], encoding="unicode")
    # Dokument-GUIDs im XML suchen
    import re
    guids = re.findall(r'OBJ_GUID="([a-f0-9]{32})"', xml_str)
    file_ids = re.findall(r'ID="([a-f0-9]{32}-\d+)"', xml_str)
    print(f"Gefundene OBJ_GUIDs: {guids}")
    print(f"Gefundene File-IDs:  {file_ids}")

In [ ]:
def get_document_metadata(doc_guid: str) -> pd.DataFrame:
    """Ruft Metadaten eines Dokuments anhand der OBJ_GUID ab."""
    root = paris_search("dokument", f'ID adj "{doc_guid}"')
    return hits_to_dataframe(get_hits(root))


def get_document_url(file_id: str) -> str:
    """Gibt die Download-URL für ein Dokument zurück (ohne /api/ Pfadbestandteil)."""
    return f"https://www.gemeinderat-zuerich.ch/dokumente/{file_id}"


# Beispiel: Metadaten eines bekannten Dokuments
# (GUID aus dem Geschäft oben, falls vorhanden)
if guids:
    df_doc_meta = get_document_metadata(guids[0])
    print("Dokument-Metadaten:")
    df_doc_meta.T
else:
    print("Keine Dokument-GUIDs gefunden.")

# Beispiel-Download-URL
if file_ids:
    download_url = get_document_url(file_ids[0])
    print(f"\nDownload-URL: {download_url}")

---
## 9. Gremien abfragen (Index: GREMIUMSUEBERSICHT)

Alle Kommissionen, Fraktionen und anderen Gremien des Gemeinderats.

In [ ]:
# Alle Sachkommissionen
root_sk = paris_search(
    "gremiumsuebersicht",
    'Gremiumstyp any "Sachkommission"',
    max_rows=100
)
print(f"Sachkommissionen: {get_num_hits(root_sk)}")

df_sk = hits_to_dataframe(get_hits(root_sk))
name_cols = [c for c in df_sk.columns if "Name" in c or "Kurzname" in c][:4]
df_sk[name_cols].head(10)

In [ ]:
# Alle Gremiumstypen
root_typen = paris_search("gremiumstyp", "seq>0", max_rows=50)
df_typen = hits_to_dataframe(get_hits(root_typen))
name_cols = [c for c in df_typen.columns if "Name" in c]
if name_cols:
    print(df_typen[name_cols[0]].tolist())
else:
    df_typen.head()

---
## 10. Paginierung: Grosse Datenmenge laden

Die API liefert maximal **1000 Treffer pro Anfrage**. Für grössere Datenmengen wird Paginierung benötigt.
Die Hilfsfunktion `load_all_pages()` (aus Kapitel 1) erledigt das automatisch.

In [ ]:
# Alle Geschäfte seit 2020 bis aktuelles Jahr laden (paginiert)
aktuelles_jahr = date.today().year
startjahr = 2020
jahres_query = " OR ".join(f'GRNr any "{j}/*"' for j in range(startjahr, aktuelles_jahr + 1))

df_alle = load_all_pages(
    index="geschaeft",
    query=jahres_query,
    page_size=500,
    max_total=3000,
)
print(f"\nGeladene Datensätze: {len(df_alle):,}")
df_alle.head(3)

In [ ]:
# Geschäftsart-Trend über Jahre
art_col  = "Geschaeft/Geschaeftsart"
grnr_col = "Geschaeft/GRNr"

if art_col in df_alle.columns and grnr_col in df_alle.columns:
    df_alle["Jahr"] = df_alle[grnr_col].str.extract(r"(\d{4})")[0]
    df_alle = df_alle.dropna(subset=["Jahr"])
    df_alle["Jahr"] = df_alle["Jahr"].astype(int)

    # Top-5 Geschäftsarten
    top5_arten = df_alle[art_col].value_counts().head(5).index.tolist()

    df_trend = (
        df_alle[df_alle[art_col].isin(top5_arten)]
        .groupby(["Jahr", art_col])
        .size()
        .unstack(art_col, fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    df_trend.plot(kind="bar", ax=ax, colormap="tab10")
    ax.set_xlabel("Jahr")
    ax.set_ylabel("Anzahl Geschäfte")
    ax.set_title(f"Trend: Top-5 Geschäftsarten {startjahr}–{aktuelles_jahr}")
    ax.legend(title="Geschäftsart", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Benötigte Spalten nicht gefunden. Verfügbar:",
          [c for c in df_alle.columns if c.count("/") <= 1][:10])


---
## 11. Praxisbeispiel: Wer hat wie viele Vorstösse eingereicht?

Kombination mehrerer Abfragen: Erstunterzeichnende pro Person auswerten.

In [ ]:
# Vorstösse 2022–2026 laden
vorst_jahre = range(2022, 2027)
query_vorstoesse = (
    "(" + " OR ".join(f'GRNr any "{y}/*"' for y in vorst_jahre) + ") "
    'AND (Geschaeftsart any "Motion" OR Geschaeftsart any "Postulat" '
    'OR Geschaeftsart any "Interpellation" OR Geschaeftsart any "Schriftliche Anfrage")'
)

root_vs = paris_search("geschaeft", query_vorstoesse, max_rows=1000)
print(f"Vorstösse 2022–2026: {get_num_hits(root_vs)}")

df_vs = hits_to_dataframe(get_hits(root_vs))

# Jahr aus GR-Nummer extrahieren
grnr_col = "Geschaeft/GRNr"
if grnr_col in df_vs.columns:
    df_vs["Jahr"] = df_vs[grnr_col].str.extract(r"(\d{4})")[0].astype(int)

# Erstunterzeichner-Spalte suchen
eu_cols = [c for c in df_vs.columns if "Erstunterzeichner" in c and "Name" in c]
print("Erstunterzeichner-Spalten:", eu_cols)


In [ ]:
if eu_cols and "Jahr" in df_vs.columns:
    eu_col = eu_cols[0]

    # Top-15 Einreichende insgesamt (2022-2026)
    top_personen = df_vs[eu_col].value_counts().head(15).index

    # Pivot: Person (Zeilen) x Jahr (Spalten)
    df_pivot = (
        df_vs[df_vs[eu_col].isin(top_personen)]
        .groupby([eu_col, "Jahr"])
        .size()
        .unstack("Jahr", fill_value=0)
    )

    # Sicherstellen, dass alle Jahre 2022-2026 vorhanden
    for j in vorst_jahre:
        if j not in df_pivot.columns:
            df_pivot[j] = 0
    df_pivot = df_pivot[sorted(df_pivot.columns)]

    # Sortierung nach Gesamtzahl (aufsteigend für horizontale Balken)
    df_pivot = df_pivot.loc[df_pivot.sum(axis=1).sort_values(ascending=True).index]

    fig, ax = plt.subplots(figsize=(10, 8))
    df_pivot.plot(kind="barh", stacked=True, ax=ax,
                  colormap="viridis", width=0.75)
    ax.set_xlabel("Anzahl Vorstösse")
    ax.set_title("Top-15 aktivste Einreichende – Vorstösse 2022–2026")
    ax.legend(title="Jahr", loc="lower right")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()
elif eu_cols:
    print("Jahr-Spalte fehlt.")
else:
    print("Erstunterzeichner-Spalte nicht gefunden.")


---
## 12. Zusammenfassung

### Wichtigste Konzepte

| Konzept | Details |
|---|---|
| **Basis-URL** | `https://www.gemeinderat-zuerich.ch/api/{index}/{funktion}` |
| **Datenformat** | XML (Namespace: `http://www.cmiag.ch/cdws/...`) |
| **Sprache** | Immer `l=de-CH` angeben |
| **Alle Objekte** | `q=seq>0` |
| **Paginierung** | `s` (Start) + `m` (Max, max. 1000) |
| **Dokument-Download** | `https://www.gemeinderat-zuerich.ch/dokumente/{file-id}` |

### CQL-Kurzreferenz

```cql
# Text-Suche
Titel any "Klimaschutz"                  # mind. ein Wort
Titel all "Klimaschutz Massnahmen"        # alle Wörter
Titel adj "Klimaschutz Massnahmen"        # exakte Phrase

# Datum
Sitzungsdatum_start >= "2024-01-01 00:00:00"
Dauer_end > "9999-12-31 00:00:00"         # offene Mandate

# Boolean
Dringlich = true
AktivesRatsmitglied = false

# Kombiniert
GRNr any "2023/*" AND Geschaeftsart any "Motion"

# Sortierung
seq>0 sortBy Name/sort.ascending
```

### Nützliche Links

- **Portal**: https://www.gemeinderat-zuerich.ch
- **API Basis-URL**: https://www.gemeinderat-zuerich.ch/api/
- **Schema-Beispiel**: https://www.gemeinderat-zuerich.ch/api/geschaeft/schema
- **OGD Datensatz**: https://data.stadt-zuerich.ch/dataset/parlamentsdienste_paris_api
- **CQL Spezifikation**: http://www.loc.gov/standards/sru/specs/cql.html